# GKP State Preparation via GRAPE

Optimal control synthesis of a pulse that prepares a finite-energy GKP $\ket{0_L}$ state from vacuum in a single Kerr-nonlinear cavity driven by two-photon (squeezing) controls. The optimization uses gradient ascent in the Fourier domain with soft Lagrange-multiplier penalties enforcing amplitude limits, smoothness, and zero-amplitude boundaries.

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import jax.numpy as jnp

from gkp_optimal_control.grape import (
    System,
    TimeGrid,
    FourierBand,
    Penalties,
    run_grape,
    save_pulse,
    forward_evolve,
)
from gkp_optimal_control.hamiltonians import (
    kerr_cavity_drift,
    kerr_cavity_squeezing_controls,
)
from gkp_optimal_control.plotting import (
    plot_photon_number,
    plot_wigner,
    set_plot_style,
)
from gkp_optimal_control.states import fock_state, gkp_states, cat_states
from gkp_optimal_control.utils import wigner_trajectory
from gkp_optimal_control.animation import animate_wigner
from gkp_optimal_control.grape_batched import bandwidth_sweep

set_plot_style()

## Target state

The target is the finite-energy logical $\ket{0_L}$ for a square-lattice GKP code with primitive displacements $\alpha = \sqrt{\pi/2}$ and $\beta = i\sqrt{\pi/2}$. The envelope width $\Delta$ sets the effective squeezing; the Fock-space truncation must accommodate the full support of the state.

In [2]:
n_fock = 80
gkp_delta = 0.3
gkp_cutoff = 10
gkp_alpha = jnp.sqrt(np.pi / 2)
gkp_beta = jnp.sqrt(np.pi / 2) * 1j

vac = fock_state(n_fock, 0)
gkp_0, gkp_1 = gkp_states(n_fock, gkp_alpha, gkp_beta, gkp_delta, gkp_cutoff)
even_cat, _ = cat_states(n_fock, gkp_alpha)

psi_init = vac.astype(jnp.complex128)
psi_targ = gkp_0.astype(jnp.complex128)

tail_population = float(jnp.sum(jnp.abs(psi_targ[-10:]) ** 2))
print(f"Target tail population (top 10 Fock levels): {tail_population:.2e}")

Target tail population (top 10 Fock levels): 9.03e-06


## System Hamiltonian

Single Kerr-nonlinear cavity in the rotating frame:
$$H/\hbar = \frac{K}{2} \hat{a}^\dagger \hat{a}^\dagger \hat{a} \hat{a} + \epsilon_I(t)\,(a^2 + a^{\dagger 2}) + i\,\epsilon_Q(t)\,(a^2 - a^{\dagger 2}).$$

The two control fields are the in-phase and quadrature components of a two-photon (squeezing) drive. Frequencies are stored as angular rates ($2\pi \times \text{MHz} = \text{rad}/\mu s$) and times in microseconds.

In [3]:
K = 2 * jnp.pi * (0.01)  # Kerr nonlinearity, rad/μs

h_drift = kerr_cavity_drift(n_fock, kerr=K)
h_controls = kerr_cavity_squeezing_controls(n_fock)  # (2, dim, dim)

system = System(
    H_drift=h_drift,
    H_controls=h_controls,
    psi_init=psi_init,
    psi_targ=psi_targ,
)

## Optimization configuration

The pulse is parametrized in the Fourier domain with a hard frequency cutoff at $f_{\max}$. Soft penalties enforce amplitude limit $\epsilon_{\max}$, smoothness (squared-derivative), and zero amplitude in the first/last `boundary_n_zero` samples.

In [4]:
time_grid = TimeGrid(T=2.0, n_steps=500)  # 1.0 μs total, dt = 2 ns

band = FourierBand(f_max=100.0)  # MHz

penalties = Penalties(
    amp=0.1,
    deriv=1e-3,
    boundary=1.0,
    eps_max=10.0,
    boundary_n_zero=3,
)

## Run GRAPE Bandwidth Sweep

L-BFGS-B with analytic gradients via JAX autodiff. The first call pays a one-time JIT compilation cost; subsequent calls reuse the compiled cost-and-gradient function.

In [11]:
import time

t0 = time.time()
small_time_grid = TimeGrid(T=1.0, n_steps=500)  # was 500
result = bandwidth_sweep(
    system,
    small_time_grid,
    f_max_values=[30.0, 80.0],  # 2 bands
    penalties=penalties,
    n_seeds=8,  # 2 seeds
    maxiter=50,
)
print(f"Done in {time.time() - t0:.1f}s")

f_max values span 2 distinct n_allowed values; running one batched call per group.

Group n_allowed=61: 1 bands × 8 seeds
Batched GRAPE: 1 bands × 8 seeds = 8 problems
  n_steps=500, param_shape=(2, 61, 2), dim=80
  Compiling + solving...
  Done in 5592.7s (699.09s per problem amortized)
  Fidelities — best/median/worst across batch: 0.5696 / 0.5690 / 0.5687
  Converged: 0/8

Group n_allowed=161: 1 bands × 8 seeds
Batched GRAPE: 1 bands × 8 seeds = 8 problems
  n_steps=500, param_shape=(2, 161, 2), dim=80
  Compiling + solving...
  Done in 1784.4s (223.05s per problem amortized)
  Fidelities — best/median/worst across batch: 0.5424 / 0.5177 / 0.4626
  Converged: 0/8
Done in 7377.6s


In [8]:
bands = [FourierBand(f_max=fm) for fm in [50.0, 80.0, 120.0]]
n_allowed_set = {b.n_allowed(time_grid) for b in bands}
print(n_allowed_set)  # check they're all equal

{321, 201, 481}
